# NB · Gold Housekeeping

> **SYNTHETIC DATA ONLY.** Reference pattern — not a certified de-identification service.

Reconciles the `gold_safe_*` tables that **exist** in the lakehouse against the ones the
accelerator **declares** in [`gold_conform.GOLD_TABLES`](../src/fabric_phi_deid/gold_conform.py),
and drops the difference.

**Why an orphaned gold table matters.** The privacy guarantees in this accelerator are
per-run and per-table: `03b` de-identifies, scans for residual PHI, and publishes in one
pass. A table left behind by an *earlier* run is outside that loop — nothing rewrote it,
so nothing re-scanned it. It was produced by whatever `deid_rules.yaml` was active at the
time, which may predate a column being added to the rulebook, and it is still bound by
the semantic model and still queryable through the SQL endpoint. It looks exactly as
trustworthy as a current table and is not. That is the gap this notebook closes.

Orphans appear routinely and quietly: an adopter runs with both sources and later drops
the Clarity extract, renames a table in `CLARITY_GOLD`, or trims `GOLD_TABLES`. The
lakehouse keeps the old tables regardless — `saveAsTable(mode="overwrite")` only touches
the names it is given.

**Modes**

| `MODE` | Drops | Use when |
|---|---|---|
| `"orphans"` | tables present but no longer declared | routine hygiene after a config change |
| `"all"` | every `gold_safe_*` table | forcing a clean rebuild after a schema change |

**Safety.** This notebook is deny-by-default: `CONFIRM = False` prints the plan and drops
nothing. It only ever touches the `gold_safe_` prefix — the de-identified, published
layer. It cannot reach `bronze_*`, `silver_*`, or `silver_deid_*`, and it never touches
the identified gold star in the RAW workspace. Re-run `03b` to rebuild anything dropped.

`MODE` and `CONFIRM` are a Fabric **parameters cell**, so a pipeline can run this on a
schedule in report-only mode and alert on orphans without granting it permission to drop
anything.

**Prerequisites**
- `src/fabric_phi_deid/` uploaded to `Files/accelerator/`
- default lakehouse bound to the workspace holding the `gold_safe_*` tables

Run order: `01` → `02` → `02b` → `03b` → `NB_scorecard`, with this notebook run
**before** `03b` whenever the gold declaration has changed.

In [ ]:
# --- make the accelerator's package importable inside Fabric ---------------------
# Driver only: this notebook runs catalog and DDL statements, no Spark UDFs, so the
# executor zip that 02b/03b need is not required here.
import os
import sys

_FILES_ROOT = "/lakehouse/default/Files/accelerator"
_SRC_CANDIDATES = [_FILES_ROOT, f"{_FILES_ROOT}/src"]
SRC_PATH = next(
    (p for p in _SRC_CANDIDATES if os.path.isdir(os.path.join(p, "fabric_phi_deid"))),
    None,
)
if SRC_PATH is None:
    raise RuntimeError(
        "fabric_phi_deid package not found. Upload src/fabric_phi_deid/ so it sits under "
        f"one of: {_SRC_CANDIDATES}"
    )
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

from fabric_phi_deid import __version__ as ENGINE_VERSION   # noqa: E402
from fabric_phi_deid import gold_conform as gc              # noqa: E402
from fabric_phi_deid.audit import get_audit_logger          # noqa: E402

log = get_audit_logger()
G = "gold_safe_"

print(f"fabric_phi_deid v{ENGINE_VERSION}  |  {len(gc.GOLD_TABLES)} gold tables declared")

In [ ]:
# ---- what this run is allowed to do ---------------------------------------------
# Fabric PARAMETERS cell: a pipeline or job run may override both values, and the
# override is injected immediately below. Deny-by-default -- leave CONFIRM False to see
# the plan, set it True to execute.
MODE = "orphans"      # "orphans" -> drop only tables no longer declared
                      # "all"     -> drop every gold_safe_ table (full rebuild)
CONFIRM = False       # nothing is dropped until this is True

In [ ]:
# ---- reconcile declared vs. present ----------------------------------------------
# Validated here rather than in the parameters cell above, because Fabric injects the
# job's overrides *after* that cell -- so a bad parameter would slip past a check there.
if MODE not in {"orphans", "all"}:
    raise ValueError(f"MODE must be 'orphans' or 'all', got {MODE!r}")
CONFIRM = bool(CONFIRM)

# gc.GOLD_TABLES is the single declaration both gold notebooks build from, so it is also
# the definition of "supposed to be here". Anything else carrying the published prefix is
# residue from an older configuration.
declared = {f"{G}{t}" for t in gc.GOLD_TABLES}
present = {t.name for t in spark.catalog.listTables() if t.name.startswith(G)}

orphaned = sorted(present - declared)
missing = sorted(declared - present)
in_sync = sorted(declared & present)


def row_count(table):
    try:
        return f"{spark.table(table).count():,}"
    except Exception as exc:                      # unreadable is itself worth reporting
        return f"<unreadable: {type(exc).__name__}>"


print(f"declared: {len(declared)}   present: {len(present)}\n")
print(f"IN SYNC ({len(in_sync)}) - declared and published")
for t in in_sync:
    print(f"    {t:44s} {row_count(t):>12s}")

print(f"\nORPHANED ({len(orphaned)}) - published but NO LONGER DECLARED")
if not orphaned:
    print("    none - the lakehouse matches the declaration")
for t in orphaned:
    print(f"    {t:44s} {row_count(t):>12s}")

print(f"\nMISSING ({len(missing)}) - declared but not published")
if not missing:
    print("    none")
for t in missing:
    print(f"    {t}")
if missing:
    print("\n    Missing tables are not an error here: a Caboodle-only or Clarity-only")
    print("    deployment legitimately publishes a subset. Run 03b to build them.")

log.info(
    f"gold housekeeping inventory: {len(in_sync)} in sync, "
    f"{len(orphaned)} orphaned, {len(missing)} missing"
)

In [ ]:
# ---- drop -------------------------------------------------------------------------
targets = orphaned if MODE == "orphans" else sorted(present)

# Safety belt. Every target is derived from a prefix filter above, so this can only fire
# if that logic is edited -- which is exactly when a stray DROP would be unrecoverable.
stray = [t for t in targets if not t.startswith(G)]
if stray:
    raise RuntimeError(
        f"refusing to run: {stray} are outside the '{G}' prefix. This notebook must "
        f"never be able to reach bronze, silver, or the identified gold star."
    )

if not targets:
    print(f"Nothing to drop in MODE={MODE!r}.")
elif not CONFIRM:
    print(f"DRY RUN - MODE={MODE!r}, CONFIRM=False. Nothing was dropped.\n")
    print(f"Would drop {len(targets)} table(s):")
    for t in targets:
        print(f"    DROP TABLE {t}")
    print("\nSet CONFIRM = True in the settings cell to execute.")
    log.info(f"gold housekeeping dry run: {len(targets)} table(s) would be dropped")
else:
    dropped = []
    for t in targets:
        spark.sql(f"DROP TABLE IF EXISTS `{t}`")
        dropped.append(t)
        print(f"    dropped {t}")
        log.info(f"gold housekeeping dropped table: {t}")
    print(f"\nDropped {len(dropped)} table(s) in MODE={MODE!r}.")
    log.info(f"gold housekeeping complete: {len(dropped)} table(s) dropped, mode={MODE}")

In [ ]:
# ---- verify ------------------------------------------------------------------------
remaining = {t.name for t in spark.catalog.listTables() if t.name.startswith(G)}
still_orphaned = sorted(remaining - declared)

print(f"{len(remaining)} {G}* table(s) remain.")
if still_orphaned:
    print(f"Still orphaned: {still_orphaned}")
    print("Expected while CONFIRM=False; otherwise investigate before publishing.")
else:
    print("No orphans: every published gold table is one the current configuration declares.")

if MODE == "all" and CONFIRM:
    print("\nGold is now empty. Re-run 03b to rebuild the star before refreshing the model.")